# 💬 Текст, пара 1. Как превратить твит в числа

> Сначала: **Файл → Сохранить копию на Диске**.

У нас 10 000 русских твитов, у каждого метка: **позитив** или **негатив**. Цель раздела — научить модель угадывать настроение твита.

Но модель умеет работать только с **числами**. Сегодня превращаем текст в числа:
1. **Токены** — режем твит на слова.
2. **Словарь** — нумеруем все слова.
3. **One-hot** — слово → вектор из нулей и одной единицы.
4. **Мешок слов (Bag of Words)** — твит → сколько раз встретилось каждое слово.

Каждое задание: замените `None` на свой код → **Shift + Enter** → запустите ячейку 🧪 под ним.

In [ ]:
#@title ⚙️ Запустите первой: библиотеки, твиты и проверки { display-mode: "form" }
# Внутри только подготовка. Читать не обязательно :)
import math, os, re, urllib.request
from collections import Counter
import numpy as np
import pandas as pd

if not os.path.exists("tweets.csv"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Scr1ptG/ml_for_3p1/main/06-text/data/tweets.csv", "tweets.csv")


def check(tests, n):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}\n   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание {n} выполнено!")
    return True


BAD = re.compile(r"бля|сук|ху[йеия]|пизд|[её]б|мудак|нах")   # мат в топах слов не показываем


print("✅ Готово. Файл tweets.csv скачан:", os.path.exists("tweets.csv"))

---
# Часть 1. Смотрим на твиты

In [ ]:
df = pd.read_csv("tweets.csv")
df.head(10)

### ✏️ Задание 1. Сколько всего твитов `n` и сколько из них позитивных `n_pos`?

In [ ]:
n = None                                   # ← len(df)
n_pos = None        # ← (df["label"] == "позитив").sum()

In [ ]:
#@title 🧪 Проверка задания 1 { display-mode: "form" }
if check([
    ('всего твитов', lambda: n == 10000, 'len(df)'),
    ('позитивных', lambda: n_pos == 5000, "(df['label'] == 'позитив').sum()"),
], 1):
    print(f"Всего {n}, позитивных {n_pos}, негативных {n - n_pos}")
    print("Классы поровну: если угадывать наугад, будет 50% правильных.")

<details><summary>💡 Подсказка</summary>

`len` считает строки таблицы. Сравнение `==` даёт True/False, а `.sum()` считает True.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
n = len(___)
n_pos = (df["label"] == "___").sum()
```
</details>

> ℹ️ Метки в этом наборе ставили **по смайликам**: `:)` — позитив, `:(` — негатив. Смайлики мы убрали, иначе было бы слишком просто. Модели придётся угадывать настроение **по словам**.

---
# Часть 2. Токены: режем текст на слова

**Токен** — кусочек текста, с которым работает модель. У нас токен = слово.

Первая идея — `split()`: резать по пробелам.

In [ ]:
t = "Отличный был денёк! Отличного вам вечера, друзья!"
print(t.split())

Плохо: `денёк!` и `денёк` — разные «слова», `Отличный` и `отличный` — тоже. Поэтому:
1. переводим в **нижний регистр**: `text.lower()`;
2. берём только **буквы**: `re.findall(r"[а-яё]+", text)` — найти все куски из русских букв.

### ✏️ Задание 2. Напишите функцию `tokenize(text)`: нижний регистр + только русские слова

**Что должно получиться:** `['отличный', 'был', 'денёк', 'отличного', 'вам', 'вечера', 'друзья']`

In [ ]:
def tokenize(text):
    return None   # ← re.findall(r"[а-яё]+", text.lower())


In [ ]:
#@title 🧪 Проверка задания 2 { display-mode: "form" }
if check([
    ('возвращает список слов', lambda: tokenize('Привет, мир!') == ['привет', 'мир'], "re.findall(r'[а-яё]+', text.lower())"),
    ('нижний регистр', lambda: tokenize('ДЕНЁК') == ['денёк'], 'Не забудьте .lower()'),
], 2):
    print(tokenize(t))
    print(tokenize(df["text"][0]))

<details><summary>💡 Подсказка</summary>

`re.findall(шаблон, строка)` возвращает список всех найденных кусков.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
return re.findall(r"[а-яё]+", text.___())
```
</details>

---
# Часть 3. Словарь: нумеруем слова

Начнём с маленького примера — 4 «твита»:

In [ ]:
corpus = [
    "Кот спит. Кот доволен",
    "Пёс спит",
    "не плохо, а хорошо",
    "не хорошо, а плохо",
]

### ✏️ Задание 3. Соберите словарь `vocab`: все **разные** слова корпуса по алфавиту

In [ ]:
all_words = []
for text in corpus:
    all_words += tokenize(text)
print(all_words)

vocab = None   # ← sorted(set(all_words))

In [ ]:
#@title 🧪 Проверка задания 3 { display-mode: "form" }
if check([
    ('только разные слова', lambda: vocab is not None and len(vocab) == len(set(vocab)) == 8, 'set(...) убирает повторы'),
    ('по алфавиту', lambda: vocab == sorted(vocab), 'sorted(...)'),
], 3):
    print("Словарь:", vocab)
    print("Размер словаря:", len(vocab))

<details><summary>💡 Подсказка</summary>

`set` — множество (без повторов), `sorted` — сортирует.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
vocab = sorted(___(all_words))
```
</details>

### ✏️ Задание 4. Сделайте `word2idx`: словарь «слово → его номер»

In [ ]:
word2idx = None   # ← {w: i for i, w in enumerate(vocab)}

In [ ]:
#@title 🧪 Проверка задания 4 { display-mode: "form" }
if check([
    ('все слова есть', lambda: word2idx is not None and len(word2idx) == len(vocab), '{w: i for i, w in enumerate(vocab)}'),
    ('номер = место в vocab', lambda: all(vocab[i] == w for w, i in word2idx.items()), 'enumerate даёт пары (номер, слово)'),
], 4):
    print(word2idx)
    print("Номер слова «спит»:", word2idx["спит"])

<details><summary>💡 Подсказка</summary>

`enumerate(vocab)` выдаёт пары `(0, 'а'), (1, 'доволен'), ...`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
word2idx = {w: i for i, w in ___(vocab)}
```
</details>

---
# Часть 4. One-hot: слово → вектор

Слово превращаем в вектор длины «размер словаря»: везде **0**, и только на месте этого слова **1**.

| | а | доволен | кот | не | пёс | плохо | спит | хорошо |
|---|---|---|---|---|---|---|---|---|
| **кот** | 0 | 0 | **1** | 0 | 0 | 0 | 0 | 0 |
| **спит** | 0 | 0 | 0 | 0 | 0 | 0 | **1** | 0 |

### ✏️ Задание 5. Функция `one_hot(word)`

In [ ]:
def one_hot(word):
    v = np.zeros(len(vocab), dtype=int)   # вектор из нулей
    # ← поставьте 1 на место word2idx[word]
    return v

In [ ]:
#@title 🧪 Проверка задания 5 { display-mode: "form" }
if check([
    ('одна единица', lambda: one_hot('кот').sum() == 1, 'v[word2idx[word]] = 1'),
    ('на нужном месте', lambda: one_hot('кот')[word2idx['кот']] == 1 and one_hot('спит')[word2idx['спит']] == 1, 'Номер берём из word2idx'),
], 5):
    for w in ["кот", "спит", "хорошо"]:
        print(f"{w:>7}", one_hot(w))

<details><summary>💡 Подсказка</summary>

Номер слова — `word2idx[word]`, по нему и ставим 1.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
v[word2idx[___]] = 1
```
</details>

### ❓ Сколько чисел нужно на одно слово в настоящих твитах?

### ✏️ Задание 6. Размер словаря всех твитов `V`

In [ ]:
real_vocab = set()
for text in df["text"]:
    real_vocab.update(tokenize(text))

V = None   # ← сколько разных слов

In [ ]:
#@title 🧪 Проверка задания 6 { display-mode: "form" }
if check([
    ('размер словаря', lambda: V == len(real_vocab) and V > 20000, 'len(real_vocab)'),
], 6):
    print(f"Разных слов: {V}")
    print(f"Значит, one-hot вектор каждого слова — {V} чисел, и из них только ОДНА единица!")

<details><summary>💡 Подсказка</summary>

Длина множества — `len(...)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
V = ___(real_vocab)
```
</details>

Вывод: one-hot — **длинный** и почти весь из нулей. Ещё он ничего не знает о смысле: «хорошо» и «отлично» так же далеки, как «хорошо» и «кот». (Это решит **word2vec** — в следующих темах.)

---
# Часть 5. Мешок слов (Bag of Words)

Твит = **сумма one-hot векторов** его слов. Получаем: сколько раз каждое слово встретилось в твите.

«Кот спит. Кот доволен» → `кот` 2 раза, `спит` 1, `доволен` 1:

| а | доволен | кот | не | пёс | плохо | спит | хорошо |
|---|---|---|---|---|---|---|---|
| 0 | 1 | **2** | 0 | 0 | 0 | 1 | 0 |

Почему «мешок»: слова высыпали в мешок и посчитали. **Порядок слов потерялся.**

### ✏️ Задание 7. Функция `bow(text)`: мешок слов для одного текста

In [ ]:
def bow(text):
    v = np.zeros(len(vocab), dtype=int)
    for w in tokenize(text):
        if w in word2idx:          # незнакомые слова пропускаем
            pass               # ← прибавьте 1 на место слова w
    return v

In [ ]:
#@title 🧪 Проверка задания 7 { display-mode: "form" }
if check([
    ('кот посчитан дважды', lambda: bow('Кот спит. Кот доволен')[word2idx['кот']] == 2, 'v[word2idx[w]] += 1'),
    ('всего 4 слова', lambda: bow('Кот спит. Кот доволен').sum() == 4, 'Прибавляйте, а не ставьте 1'),
    ('незнакомые слова не ломают', lambda: bow('кот и собака').sum() == 1, 'Проверка if w in word2idx уже есть'),
], 7):
    print(vocab)
    for text in corpus:
        print(bow(text), "←", text)

<details><summary>💡 Подсказка</summary>

Как в one-hot, только `+= 1` вместо `= 1`: слово может встретиться несколько раз.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
v[word2idx[w]] ___ 1
```
</details>

### ✏️ Задание 8. Проверьте: одинаковы ли мешки у «не плохо, а хорошо» и «не хорошо, а плохо»?

In [ ]:
same = None   # ← np.array_equal(bow(corpus[2]), bow(corpus[3]))

In [ ]:
#@title 🧪 Проверка задания 8 { display-mode: "form" }
if check([
    ('сравнили', lambda: same == True, 'np.array_equal(a, b) — True, если массивы совпадают'),
], 8):
    print(corpus[2], "→", bow(corpus[2]))
    print(corpus[3], "→", bow(corpus[3]))
    print("Смысл противоположный, а векторы одинаковые: мешок слов не помнит порядок.")

<details><summary>💡 Подсказка</summary>

`np.array_equal(x, y)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
same = np.array_equal(bow(corpus[2]), bow(corpus[___]))
```
</details>

---
# Часть 6. То же самое одной строкой: `CountVectorizer`

В библиотеке scikit-learn мешок слов уже написан. Сравним с нашим.

### ✏️ Задание 9. Постройте мешок слов для `corpus` через `CountVectorizer`

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(token_pattern=r"[а-яё]+")   # наш же шаблон слов
X = None   # ← cv.fit_transform(corpus): выучить словарь и посчитать

In [ ]:
#@title 🧪 Проверка задания 9 { display-mode: "form" }
if check([
    ('словарь как у нас', lambda: list(cv.get_feature_names_out()) == vocab, 'cv.fit_transform(corpus)'),
    ('матрица как у нас', lambda: np.array_equal(X.toarray(), np.array([bow(t) for t in corpus])), 'Строка = твит, столбец = слово'),
], 9):
    print(cv.get_feature_names_out())
    print(X.toarray())
    print("Совпало с нашим bow() 🎉")

<details><summary>💡 Подсказка</summary>

`fit` — выучить словарь, `transform` — посчитать слова. `fit_transform` делает оба шага.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
X = cv.___(corpus)
```
</details>

### ✏️ Задание 10. Мешок слов для **всех** твитов: матрица `X_all`

In [ ]:
cv_all = CountVectorizer(token_pattern=r"[а-яё]+")
X_all = None   # ← как в прошлом задании, только df["text"]

In [ ]:
#@title 🧪 Проверка задания 10 { display-mode: "form" }
if check([
    ('строк = твитов', lambda: X_all.shape[0] == 10000, "cv_all.fit_transform(df['text'])"),
    ('столбцов = слов', lambda: X_all.shape[1] == V, 'Тот же шаблон слов'),
], 10):
    zeros = 1 - X_all.nnz / (X_all.shape[0] * X_all.shape[1])
    print("Размер матрицы:", X_all.shape)
    print(f"Нулей в матрице: {zeros:.2%} — поэтому sklearn хранит её «сжато» (sparse)")

<details><summary>💡 Подсказка</summary>

Подставьте весь столбец `df["text"]`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
X_all = cv_all.fit_transform(df["___"])
```
</details>

---
# Часть 7. Какие слова у позитива, а какие у негатива?

Сложим мешки всех позитивных твитов и всех негативных — получим, сколько раз каждое слово встречалось в каждом классе.

### ✏️ Задание 11. Посчитайте частоты слов в классах: `pos_counts` и `neg_counts`

In [ ]:
words = cv_all.get_feature_names_out()
is_pos = (df["label"] == "позитив").values

pos_counts = np.asarray(X_all[is_pos].sum(axis=0)).ravel()     # сумма по позитивным твитам
neg_counts = None   # ← то же для ~is_pos (НЕ позитив)

In [ ]:
#@title 🧪 Проверка задания 11 { display-mode: "form" }
if check([
    ('размер', lambda: len(neg_counts) == len(words), 'np.asarray(X_all[~is_pos].sum(axis=0)).ravel()'),
    ('слов в негативе', lambda: neg_counts.sum() == X_all[~is_pos].sum(), '~is_pos — все НЕ позитивные'),
], 11):
    score = (pos_counts + 1) / (neg_counts + 1)
    freq = (pos_counts + neg_counts >= 20) & np.array([not BAD.search(w) for w in words])
    top_pos = [words[i] for i in np.argsort(-score) if freq[i]][:15]
    top_neg = [words[i] for i in np.argsort(score) if freq[i]][:15]
    print("😊 чаще в позитиве:", ", ".join(top_pos))
    print("😞 чаще в негативе:", ", ".join(top_neg))

<details><summary>💡 Подсказка</summary>

`~` переворачивает True/False. Остальное — как в строке с `pos_counts`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
neg_counts = np.asarray(X_all[___].sum(axis=0)).ravel()
```
</details>

### ✅ Итог пары
| Шаг | Что делает | Минус |
|---|---|---|
| токенизация | текст → список слов | «денёк» и «денька» — разные слова |
| словарь | слово → номер | десятки тысяч слов |
| one-hot | слово → вектор с одной 1 | огромный, не знает смысла |
| мешок слов | твит → сколько раз какое слово | теряет порядок, частые слова («и», «я») «кричат» громче всех |

На следующей паре: **TF-IDF** — как приглушить частые слова, и **обучим модель**, которая отличает позитив от негатива.